# Part 2 — Our First Agent-Based Model: Desire Paths

**Recap from Part 1:** for "organized" problems (an ecosystem), the classical approach was to write equations for the whole *group* — no individuals. It worked, but only because we already knew the right equations, and every individual was treated as identical.

**Now we try something completely different.** You've probably seen a "desire path" in real life: a dirt trail worn into the grass where people kept cutting the corner instead of using the paved sidewalk. Nobody planned that trail. Nobody wrote an equation for it. It just **emerged**, because:

- people who walk somewhere leave a small trace behind (worn grass),
- untouched grass grows back over time,
- and once a shortcut gets worn in *enough*, people start noticing and using it on purpose — which wears it in even more.

That's it. Three simple, local rules. No individual "knows" a path is forming — but a path forms anyway. This is the essence of **Agent-Based Modeling (ABM)**: give individuals simple personal rules, let them interact, and watch structure emerge on its own — nobody tells the simulation in advance what the final pattern should look like.

We'll build this step by step. Again: don't worry about reading the code closely — just run each cell and read what it's doing.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = False

# Our "world" is a 50x50 grid, like a park divided into small squares.
# The edges wrap around (walk off the right edge, you reappear on the left)
# just so nobody gets stuck in a corner.
WORLD_WIDTH, WORLD_HEIGHT = 50, 50

print("World created: 50 x 50 grid.")


## Building the model, one small rule at a time

We'll define a handful of very small helper pieces first, each one doing exactly one job. Then at the end we'll combine them into the full simulation.


In [ ]:
# Helper 1: measuring distance on a world that wraps around at the edges.
# (If this seems fiddly, don't worry -- all it does is measure "how far apart
# are these two points," accounting for the fact that the world wraps around.)

def wrap_delta(a, b, size):
    d = b - a
    if d > size / 2: d -= size
    if d < -size / 2: d += size
    return d

def distance(p1, p2, width, height):
    dx = wrap_delta(p1[0], p2[0], width)
    dy = wrap_delta(p1[1], p2[1], height)
    return np.hypot(dx, dy)

print("Helper 1 ready: we can now measure distances.")


**Rule 1 — walking leaves a trace.** Every time a walker crosses a patch of ground, that patch becomes a little bit more "popular." If a patch's popularity crosses a threshold, it permanently becomes an official **route** (we'll color these gray, like a paved path).

**Rule 2 — unused ground recovers.** Any patch nobody is currently standing on slowly loses popularity over time (like grass growing back). If popularity drops low enough, a patch stops being a route.

**Rule 3 — people prefer existing paths.** If a walker notices a nearby route that would meaningfully shorten their trip, they head for it instead of cutting straight across open ground.

That's the entire "mind" of a walker. Let's write it as one function that simulates the whole thing, tick by tick.


In [ ]:
def run_desire_path_simulation(n_walkers, route_threshold, destinations,
                                 popularity_gain=2.0, decay_rate=2.0,
                                 vision=4, n_ticks=400, seed=0):
    """
    n_walkers        -- how many little people are walking around
    route_threshold  -- how much popularity a patch needs before it becomes
                        a permanent path (set this absurdly high to turn OFF
                        path formation entirely)
    destinations     -- fixed points people walk between (e.g. building doors).
                        Leave this EMPTY to make everyone wander to random
                        spots forever, with no real destinations.
    """
    rng = np.random.default_rng(seed)
    has_destinations = len(destinations) >= 2

    popularity = np.zeros((WORLD_WIDTH, WORLD_HEIGHT))   # Rule 1 & 2 tracker
    is_route   = np.zeros((WORLD_WIDTH, WORLD_HEIGHT), dtype=bool)  # permanent paths
    footfall   = np.zeros((WORLD_WIDTH, WORLD_HEIGHT))    # just for us to inspect later

    positions = rng.uniform(0, WORLD_WIDTH, size=(n_walkers, 2))
    if has_destinations:
        goals = np.array([destinations[rng.integers(0, len(destinations))] for _ in range(n_walkers)], dtype=float)
    else:
        goals = rng.uniform(0, WORLD_WIDTH, size=(n_walkers, 2))

    route_count_over_time = np.zeros(n_ticks)

    for tick in range(n_ticks):
        occupied_patches = set()

        for w in range(n_walkers):
            x, y = positions[w]
            patch_x, patch_y = int(x) % WORLD_WIDTH, int(y) % WORLD_HEIGHT
            occupied_patches.add((patch_x, patch_y))

            # did this walker just reach their goal? if so, pick a new one.
            gx, gy = goals[w]
            if distance((x, y), (gx, gy), WORLD_WIDTH, WORLD_HEIGHT) < 1.0:
                if has_destinations:
                    goals[w] = np.array(destinations[rng.integers(0, len(destinations))], dtype=float)
                else:
                    goals[w] = rng.uniform(0, WORLD_WIDTH, size=2)
                gx, gy = goals[w]

            # --- RULE 1: walking here boosts this patch's popularity ---
            footfall[patch_x, patch_y] += 1
            if not is_route[patch_x, patch_y]:
                popularity[patch_x, patch_y] += popularity_gain
                if popularity[patch_x, patch_y] >= route_threshold:
                    is_route[patch_x, patch_y] = True   # this patch just became a permanent path!

            # --- RULE 3: is there a nearby route that shortens the trip? ---
            my_distance_to_goal = distance((x, y), (gx, gy), WORLD_WIDTH, WORLD_HEIGHT)
            best_target, best_target_distance = None, None
            for cx in range(int(x) - vision, int(x) + vision + 1):
                for cy in range(int(y) - vision, int(y) + vision + 1):
                    cxm, cym = cx % WORLD_WIDTH, cy % WORLD_HEIGHT
                    if not is_route[cxm, cym]:
                        continue
                    dist_to_me = np.hypot(cx - x, cy - y)
                    if dist_to_me > vision:
                        continue
                    dist_to_goal_from_there = distance((cx, cy), (gx, gy), WORLD_WIDTH, WORLD_HEIGHT)
                    if dist_to_goal_from_there < my_distance_to_goal - 1:
                        if best_target_distance is None or dist_to_me < best_target_distance:
                            best_target_distance = dist_to_me
                            best_target = (cx, cy)

            # walk one step toward whichever target we picked (a route, or the goal itself)
            target = best_target if best_target is not None else (gx, gy)
            dx = wrap_delta(x, target[0], WORLD_WIDTH)
            dy = wrap_delta(y, target[1], WORLD_HEIGHT)
            step_size = np.hypot(dx, dy)
            if step_size > 1e-6:
                positions[w, 0] = (x + dx / step_size) % WORLD_WIDTH
                positions[w, 1] = (y + dy / step_size) % WORLD_HEIGHT

        # --- RULE 2: unused patches slowly lose their popularity ---
        untouched = np.ones((WORLD_WIDTH, WORLD_HEIGHT), dtype=bool)
        for (px, py) in occupied_patches:
            untouched[px, py] = False
        popularity[untouched] *= (100 - decay_rate) / 100
        is_route[(popularity < 1) & untouched] = False   # a path can fade away if abandoned

        route_count_over_time[tick] = is_route.sum()

    return route_count_over_time, footfall, is_route

print("Full simulation defined. It combines the 3 rules above, run one tick at a time.")


## Let's run three versions of the same little world

Same grid, same number of walkers (60), same number of time steps (400) every time. We only change **whether real destinations exist** and **whether Rule 3 (path formation) is switched on**:

- **A — Nobody has a real destination.** Everyone just wanders to a fresh random spot forever. This is our "gas of particles" scenario again, but with walking agents instead.
- **B — Real destinations exist, but path-forming is switched off.** People walk between 5 fixed buildings, but no patch is ever allowed to become an official route. This isolates: how much structure comes just from *having* destinations, with no self-reinforcement at all?
- **C — Real destinations, AND path-forming switched on.** The full model. Watch what happens.


In [ ]:
# 5 fixed "buildings" -- imagine these as doors on a campus
buildings = [(8, 8), (42, 8), (8, 42), (42, 42), (25, 25)]

scenario_A = dict(destinations=[],        route_threshold=1e9)   # no real destinations, paths impossible
scenario_B = dict(destinations=buildings, route_threshold=1e9)   # real destinations, paths impossible
scenario_C = dict(destinations=buildings, route_threshold=40)    # real destinations, paths allowed

results = {}
for label, settings in [("A. No real destinations", scenario_A),
                         ("B. Destinations, no path-forming", scenario_B),
                         ("C. Destinations + path-forming (full model)", scenario_C)]:
    route_history, footfall, is_route = run_desire_path_simulation(
        n_walkers=60, n_ticks=400, seed=3, **settings)
    results[label] = dict(route_history=route_history, footfall=footfall, is_route=is_route)
    print(f"{label}: {int(route_history[-1])} patches became permanent paths")


In [ ]:
# Let's SEE it. Brighter = walked over more often. Cyan dots = permanent paths.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, label in zip(axes, results):
    footfall = results[label]["footfall"]
    ax.imshow(np.log1p(footfall).T, origin="lower", cmap="inferno")
    is_route = results[label]["is_route"]
    ry, rx = np.where(is_route.T)
    if len(rx):
        ax.scatter(rx, ry, s=2, c="cyan")
    if "A." not in label:
        bx, by = zip(*buildings)
        ax.scatter(bx, by, marker="s", s=60, c="white", edgecolor="black")
    ax.set_title(label, fontsize=10)
    ax.axis("off")
plt.suptitle("Where people walked (brighter = more traffic). Cyan = permanent paths that formed.")
plt.tight_layout()
plt.show()


**What to notice:** Scenario A is just diffuse noise -- no real structure. Scenario B already shows some concentration near the buildings (just from having fixed destinations) but no permanent structure -- turn the walkers off and every trace of it would vanish, since popularity was never allowed to lock in. Scenario C is different in kind, not just degree: real paths have permanently formed (the cyan dots), and they'll keep shaping how *future* walkers move, long after any particular walker has moved on.


## The big question: if we ran this again, would we get the same thing?

This is really the heart of the matter. Let's run Scenario A (no destinations) and Scenario C (full model) **four separate times each**, changing nothing except the random luck of exactly where each walker starts and which way they first wander. Everything else -- number of walkers, number of buildings, all the rules -- stays identical.

**Question:** does the *big-picture result* come out the same every time, or not?


In [ ]:
def traffic_spread(footfall):
    """A single number describing how SPREAD OUT the foot traffic is.
    Big number = traffic is spread evenly everywhere.
    Small number = traffic is concentrated onto just a few spots."""
    p = footfall.flatten()
    p = p[p > 0]
    p = p / p.sum()
    return -(p * np.log(p)).sum()

seeds_to_try = [1, 2, 3, 4]

scenario_A_spread = []
scenario_C_route_counts = []
scenario_C_route_locations = []

for seed in seeds_to_try:
    _, footfall_A, _ = run_desire_path_simulation(n_walkers=60, n_ticks=400, seed=seed, **scenario_A)
    scenario_A_spread.append(traffic_spread(footfall_A))

    route_history_C, _, is_route_C = run_desire_path_simulation(n_walkers=60, n_ticks=400, seed=seed, **scenario_C)
    scenario_C_route_counts.append(route_history_C[-1])
    scenario_C_route_locations.append(set(zip(*np.where(is_route_C))))

print("Scenario A (no destinations) -- 'how spread out is traffic', across 4 different runs:")
print(" ", [f"{s:.4f}" for s in scenario_A_spread])

print()
print("Scenario C (full model) -- 'how many permanent paths formed', across 4 different runs:")
print(" ", [int(c) for c in scenario_C_route_counts])


In [ ]:
# Let's put a number on "how much did the result vary, run to run?"

spread_range = max(scenario_A_spread) - min(scenario_A_spread)
spread_pct = 100 * spread_range / np.mean(scenario_A_spread)

count_range = max(scenario_C_route_counts) - min(scenario_C_route_counts)
count_pct = 100 * count_range / np.mean(scenario_C_route_counts)

print(f"Scenario A: the 'traffic spread' number varies by only {spread_pct:.2f}% across 4 runs.")
print(f"            -> basically the SAME answer every single time.")
print()
print(f"Scenario C: the 'number of paths formed' varies by {count_pct:.0f}% across 4 runs.")
print(f"            -> a genuinely DIFFERENT answer almost every time.")


In [ ]:
# One more check for Scenario C: even when two runs form a SIMILAR number of
# paths, are they the SAME specific paths? Or did history send them down
# different routes entirely?

print("How much do different runs agree on WHICH exact spots became paths?")
for i in range(len(seeds_to_try)):
    for j in range(i + 1, len(seeds_to_try)):
        a, b = scenario_C_route_locations[i], scenario_C_route_locations[j]
        overlap = len(a & b) / len(a | b)
        print(f"  run {seeds_to_try[i]} vs run {seeds_to_try[j]}: {overlap*100:.0f}% of the same spots became paths")


**What just happened:** in Scenario A, every run gives basically the identical answer for the group-level statistic — even though we can't predict any single walker. That's Weaver's "disorganized complexity" idea exactly: individuals are unpredictable, the group average isn't.

In Scenario C, the group-level *outcome itself* changes substantially every time — different number of paths, and mostly different specific paths, even though nothing about the rules or the setup changed. That's the signature of Weaver's "organized complexity": the system has *memory* (whichever spot happens to get popular first shapes everything that follows), so there's no fixed formula that tells you the answer in advance. You genuinely have to run the simulation and watch history unfold.

**And notice what we never did, in this entire notebook: we never wrote a single equation describing the whole population.** Every walker just followed 3 small personal rules. The paths, the traffic patterns, the unpredictability — all of it emerged on its own. That's Agent-Based Modeling.


## Recap

| | Scenario A (no destinations) | Scenario C (full model) |
|---|---|---|
| individual behavior | unpredictable | unpredictable |
| group-level result, run to run | almost identical every time | substantially different every time |
| does the past shape the future? | no | yes -- whichever spot gets popular first stays popular |
| did we need to know the "right" equations in advance? | no | no |

That last row is the whole point of this course. In Part 1, "organized complexity" (the ecosystem) only worked because we already knew the right group-level equations. Here, we didn't need to know anything about the group in advance — we only needed simple rules for individuals, and the organization **emerged** by itself.
